# **TUTORIAL**: Study objective B1 (uncertainty)

AESA Phase B: allocated shares of carrying capacities.\
Run aSoCC Monte Carlo **uncertainty** on deterministic allocated shares of carrying capacities (aSoCC).\
Workflows can include pyaesa computed aSoCC methods and staged external user
provided aSoCC methods.

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommend to first go through all notebooks available in the `tutorials/core_prerequisites` folder, to have extensive details on what the functions do.

### Functional units and selectors

The example in this tutorial use the **functional unit** `L2.c.b` with the following **selectors**: one producing sector `s_p="Paper"` and one consuming region `r_c="FR"`.

For more details on functional units, please check out:
- the short guide to functional units, region/sector selectors and allocation methods at `tutorials/study_objectives/1_functional_units_and_allocation_methods.md` for a streamlined explanation.
- the dedicated complete documentation in the methodological notes at `methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf` for an in-depth explanation.

In [ ]:
project_name_ = "SO_B1_demo_paper_fr"

source_ = "exiobase_3102_ixi"
fu_code_ = "L2.c.b"
study_period_ = range(2010,2021)
s_p_ = ["Paper"]
r_c_ = ["FR"]
lcia_method_ = "pb_lcia" 

# Basic features of the uncertainty function

### In a nutshell

The function takes necessary **inputs**, for two main purposes:
- inputs for the deterministic function `base_asocc_args`, which mostly includes:
    - usual inputs parameters, such as `project_name`, `source`, `years`, `lcia_method`, `fu_code` and the relevant selectors (here `s_p` and `r_c`) for the selected functional unit.\
    - All parameters for allocation methods and specific seetings related to enacting metrics can also be reached through nested keys.
- inputs for the configuration of uncertainty analysis (Monte Carlo) through `uncertainty_config`, which mostly includes:
    - `mc_parameters`, `lcia_uncertainty`, and `inter_method_uncertainty` parameters, with nested keys.
    - `sobol_parameters` parameter, with nested keys.

The deterministic **output** folder contains:
- Monte Carlo run values, summary statistics, and Sobol variance decomposition outputs for uncertainty source contributions when requested. Sobol analysis writes `README_sobol.txt` with the results for interpretation.
- **Figures** are rendered when requested, as controlled with the `figures` and `figure_format` parameters. One additional parameter `figure_options` allows to restrain figures generation to a subset of figures families.
- log files, including uncertainty source parameter logs.

Basic features also involve:
- **grouping** of sectors and/or regions: use the `group_sec`, `group_reg`, and `group_version` parameters.
- **overwriting** of existing values: use the `refresh` parameter.

The uncertainty source note is
`data_raw/methodological_notes/methodological_note__acc_uncertainty_sources.pdf`.

For LCIA uncertainty, sector coefficient of variation files are under
`data_raw/mrio/exiobase_3/lcia/carbon_accounts_covs/`.
 

### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>uncertainty_asocc(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_asocc_args</code></td><td>Deterministic aSoCC selector envelope. Write nested<br>&#10;arguments as <code>base_asocc_args={&quot;project_name&quot;: &quot;...&quot;, &quot;source&quot;: &quot;...&quot;, &quot;fu_code&quot;: &quot;...&quot;}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>project_name</code>: Required project name used to build<br>&#10;  <code>&lt;repo&gt;/&lt;project_name&gt;</code>.<br>&#10;&bull; <code>source</code>: MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_3102_pxp&quot;</code>, or <code>&quot;oecd_v2025&quot;</code>), or <code>&quot;iso3&quot;</code><br>&#10;  for ISO3 only mode (L1 EG/PR(GDPcap) only).<br>&#10;<span style="color:#c45f00">&bull; <code>group_reg</code>: If <code>True</code>, aggregate regions using a grouping<br>&#10;  file. <strong>Default</strong> <code>False</code> keeps native source regions.</span><br>&#10;<span style="color:#c45f00">&bull; <code>group_sec</code>: If <code>True</code>, aggregate sectors using a grouping<br>&#10;  file. <strong>Default</strong> <code>False</code> keeps native source sectors.</span><br>&#10;<span style="color:#c45f00">&bull; <code>group_version</code>: Grouping version tag used to resolve the<br>&#10;  region/sector mapping CSVs. Required when <code>group_reg</code> or<br>&#10;  <code>group_sec</code> is True. <strong>Defaults to</strong> an empty string for ungrouped<br>&#10;  processing. Follow <code>README_grouping.txt</code> in the active<br>&#10;  <code>data_raw/mrio/&lt;source&gt;/grouping</code> folder to name grouping<br>&#10;  versions and place the matching mapping CSVs.</span><br>&#10;&bull; <code>years</code>: Studied years. Accepts a single year, list, or<br>&#10;  range. <strong>If omitted</strong>, all available MRIO years for the selected<br>&#10;  source/group version are used.<br>&#10;&bull; <code>fu_code</code>: Required functional unit code (for example<br>&#10;  <code>&quot;L1.a&quot;</code>, <code>&quot;L2.c.b&quot;</code>). See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for all available functional unit codes and the system<br>&#10;  boundaries each represents.<br>&#10;&bull; <code>s_p</code>: Producing sector filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing sectors. To<br>&#10;  identify valid sector names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/.../group_sec_template.csv</code><br>&#10;  file. For EXIOBASE sector definitions, see<br>&#10;  <code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>;<br>&#10;  EXIOBASE ixi and pxp use different sector lists.<br>&#10;&bull; <code>r_p</code>: Producing region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/group_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull; <code>r_c</code>: Consuming region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid consuming regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/group_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull; <code>r_f</code>: Final demand region filter(s), single string or list.<br>&#10;  If this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid final demand regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/group_reg_template.csv</code><br>&#10;  file.<br>&#10;<span style="color:#087f5b">&bull; <code>aggreg_indices</code>: Whether multiple selected region/sector<br>&#10;  indices are reported as separate rows or summed into one row<br>&#10;  after the selected MRIO scope is computed.<br>&#10;  &bull; <code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;  &bull; <code>True</code>: sum selected values into one row.<br>&#10;  Not allowed for <code>L2.a.b</code>, <code>L2.b.b</code>, and <code>L2.c.b</code> because<br>&#10;  aggregating CBA total demand system boundaries can double count.<br>&#10;  For these functional units, define the aggregation from<br>&#10;  <code>process_mrio(...)</code> onward with<br>&#10;  <code>group_reg</code>/<code>group_sec</code>/<code>group_version</code>.</span><br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>method_plan</code>: <code>method_plan</code> <strong>defaults to</strong> <code>&quot;default&quot;</code> and<br>&#10;  accepts <code>&quot;default&quot;</code>, <code>&quot;one_step&quot;</code>, <code>&quot;two_steps&quot;</code>,<br>&#10;  <code>&quot;pairs&quot;</code>, or <code>&quot;one_step_pairs&quot;</code>. <strong>When omitted</strong>, all pyaesa<br>&#10;  allocation methods available for the selected <code>fu_code</code> are<br>&#10;  applied. See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for the allocation methods available per functional unit,<br>&#10;  including definitions and mathematical expressions.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l1_methods</code>: Optional L1 subset. <strong>Omit</strong> it to keep all L1<br>&#10;  methods allowed by <code>method_plan</code>. In <code>&quot;default&quot;</code>, this<br>&#10;  filters only L1 weights used by two step methods. In<br>&#10;  <code>&quot;two_steps&quot;</code>, it filters the two step cartesian L1 side.</span><br>&#10;<span style="color:#087f5b">&bull; <code>one_step_methods</code>: Optional one step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all one step methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull; <code>two_step_methods</code>: Optional two step L2 subset. <strong>Omit</strong> it to<br>&#10;  keep all two step L2 methods allowed by <code>method_plan</code>.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l1_l2_pairs</code>: Explicit pair list formatted as<br>&#10;  <code>&quot;L1METHOD::L2METHOD&quot;</code>. <strong>Omit</strong> it unless <code>method_plan</code> is<br>&#10;  <code>&quot;pairs&quot;</code> or <code>&quot;one_step_pairs&quot;</code>.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l1_reg_aggreg</code>: L1 aggregation mode for methods where timing<br>&#10;  matters (<code>PR(GDPcap)</code>, <code>PR-HR(Ecap)</code> and <code>AR(Ecap)</code>).<br>&#10;  <code>&quot;pre&quot;</code> aggregates regions before L1 computation. <code>&quot;post&quot;</code><br>&#10;  (<strong>default</strong>) computes on original regions, then aggregates.</span><br>&#10;<span style="color:#c45f00">&bull; <code>lcia_method</code>: LCIA method(s) selected for LCIA based<br>&#10;  allocation methods (acquired rights (AR) methods at L1 and L2<br>&#10;  and historical responsibility (PR-HR) at L1). Options are for<br>&#10;  example <code>&quot;pb_lcia&quot;</code> or <code>[&quot;pb_lcia&quot;, &quot;gwp100_lcia&quot;]</code>.<br>&#10;  <code>None</code> skips LCIA characterization and excludes LCIA based<br>&#10;  allocation methods. <strong>Defaults to</strong> <code>None</code>. pyaesa currently<br>&#10;  supports LCIA based allocation methods only for EXIOBASE<br>&#10;  sources. To add a custom LCIA method with which run<br>&#10;  <code>process_mrio(...)</code>, follow<br>&#10;  <code>README_add_custom_lcia_characterization_matrices.txt</code> in<br>&#10;  <code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/</code><br>&#10;  and pass the custom method file stem here.</span><br>&#10;<span style="color:#087f5b">&bull; <code>reference_years</code>: Acquired rights (AR) methods reference<br>&#10;  year selector. Accepts a single year, list, or range. If<br>&#10;  <strong>omitted</strong>, AR routes use all historical years in the studied range<br>&#10;  up to the source registry historical cutoff. For EXIOBASE<br>&#10;  3.10.2 and OECD ICIO v2025, the cutoff is 2022; other supported<br>&#10;  MRIO sources use their own registry cutoff.</span><br>&#10;<span style="color:#087f5b">&bull; <code>ssp_scenario</code>: SSP scenario name or list. <strong>Defaults to</strong><br>&#10;  <code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code> and is applied<br>&#10;  when scenario dependent inputs are required.</span><br>&#10;<span style="color:#087f5b">&bull; <code>projection_mode</code>: Projection policy for post historical<br>&#10;  years of L2 utilitarian (UT) methods (MRIO economic enacting<br>&#10;  metrics). <strong>Defaults to</strong> <code>&quot;regression&quot;</code>. <code>&quot;regression&quot;</code><br>&#10;  projects UT inputs for future years. <code>&quot;historical_reuse&quot;</code><br>&#10;  reuses historical UT structures.</span><br>&#10;<span style="color:#087f5b">&bull; <code>reg_window</code>: Historical regression fit window for regression<br>&#10;  mode. Provide it as <code>range(start_year, end_year + 1)</code> or as<br>&#10;  an explicit list of consecutive years in fit window order. When<br>&#10;  <strong>omitted</strong>, the source registry supplies the <strong>default</strong> fit window<br>&#10;  from the modeled year minimum through the source historical<br>&#10;  cutoff. For EXIOBASE 3.10.2 and OECD ICIO v2025, this resolves<br>&#10;  to 1995 to 2022; other supported MRIO sources use their own<br>&#10;  registry window.</span><br>&#10;<span style="color:#087f5b">&bull; <code>l2_reuse_years</code>: Historical L2 reuse year selector used by<br>&#10;  all UT historical reuse routes. In<br>&#10;  <code>projection_mode=&quot;historical_reuse&quot;</code> it applies to all UT<br>&#10;  methods; in <code>projection_mode=&quot;regression&quot;</code> it applies to<br>&#10;  adjusted UT routes (<code>UT(FDa)</code>, <code>UT(GVAa)</code>), which always<br>&#10;  use historical reuse as regression is not supported (would<br>&#10;  require regression on full MRIO). <strong>If omitted</strong>, <strong>defaults to</strong><br>&#10;  <code>reg_window</code> when required.</span></td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>uncertainty_config</code></td><td>Monte Carlo configuration dictionary. The <strong>default</strong><br>&#10;signature activates projection, reference year, and inter method<br>&#10;uncertainty. LCIA uncertainty is inactive by <strong>default</strong> because L2 LCIA<br>&#10;rows require <code>sector_cov_mapping</code>: keys are output <code>s_p</code><br>&#10;labels and values are sector CoV codes from <code>sec_cbca_covs.csv</code>.<br>&#10;Inter MRIO uncertainty is inactive by <strong>default</strong> because it requires<br>&#10;an alternate published disaggregated aSoCC source. Source blocks<br>&#10;use an <code>active</code> boolean; write <code>active=False</code> to disable a<br>&#10;<strong>default</strong> active source. See<br>&#10;<code>data_raw/methodological_notes/methodological_note__acc_uncertainty_sources.pdf</code><br>&#10;for uncertainty source definitions and mathematical expressions.<br>&#10;&#160;<br>&#10;Accepted keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>mc_parameters</code>: optional dictionary with <code>convergence</code> and<br>&#10;  <code>fixed</code> mode blocks. Exactly one mode must be active.<br>&#10;&#160;<br>&#10;  Nested mode blocks:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>convergence</code>: convergence mode block. This is the <strong>default</strong><br>&#10;    active mode.<br>&#10;&#160;<br>&#10;    Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">    &bull; <code>active</code>: Whether convergence mode is active.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>max_runs</code>: Maximum number of Monte Carlo runs allowed<br>&#10;      before stopping.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>rtol</code>: Relative tolerance used to decide whether<br>&#10;      monitored summary statistics have converged.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>stable_runs</code>: Number of consecutive accepted runs that<br>&#10;      must remain within tolerance before the run stops.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>convergence_statistics</code>: Statistic monitored for<br>&#10;      convergence. Monte Carlo convergence is mean only and<br>&#10;      <strong>defaults to</strong> <code>[&quot;mean&quot;]</code>.</span><br>&#10;&#160;<br>&#10;<span style="color:#c45f00">  &bull; <code>fixed</code>: fixed run count mode block.<br>&#10;&#160;<br>&#10;    Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">    &bull; <code>active</code>: Whether fixed mode is active.</span><br>&#10;    &bull; <code>n_runs</code>: Exact number of Monte Carlo runs.<br>&#10;&#160;<br>&#10;<span style="color:#c45f00">&bull; <code>lcia_uncertainty</code>: optional LCIA source block. It <strong>defaults<br>&#10;  to</strong> <code>{&quot;active&quot;: False, &quot;sector_cov_mapping&quot;: {}}</code>. Country<br>&#10;  level LCIA CoVs are resolved automatically. L2 sector resolved<br>&#10;  LCIA rows require <code>sector_cov_mapping</code> to map output <code>s_p</code><br>&#10;  labels to sector CoV codes from <code>sec_cbca_covs.csv</code>, for example<br>&#10;  <code>{&quot;active&quot;: True, &quot;sector_cov_mapping&quot;: {&quot;Paper&quot;: &quot;Paper&quot;}}</code>.<br>&#10;  Carbon consumption based accounts<br>&#10;  coefficients of variation (CoV) files are available under<br>&#10;  <code>data_raw/mrio/exiobase_3/lcia/carbon_accounts_covs/</code>. Users<br>&#10;  can inspect <code>sec_cbca_covs.csv</code> for sector CoV codes before<br>&#10;  choosing <code>sector_cov_mapping</code> values.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#c45f00">  &bull; <code>active</code>: Whether LCIA uncertainty is active.</span><br>&#10;<span style="color:#c45f00">  &bull; <code>sector_cov_mapping</code>: Mapping from output <code>s_p</code></span><br>&#10;    labels to sector CoV codes from <code>sec_cbca_covs.csv</code>.<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>projection_uncertainty</code>: optional source block. It <strong>defaults<br>&#10;  to</strong> <code>{&quot;active&quot;: True}</code>. For prospective rows using L2<br>&#10;  historical reuse, each Monte Carlo run samples one L2 reuse<br>&#10;  year uniformly from the deterministic <code>l2_reuse_years</code><br>&#10;  candidates requested for the years where reuse applies.<br>&#10;&#160;<br>&#10;  Nested key:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether projection uncertainty is active.</span><br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>reference_year_uncertainty</code>: optional source block. It<br>&#10;  <strong>defaults to</strong> <code>{&quot;active&quot;: True}</code>. For acquired rights (AR)<br>&#10;  routes, each Monte Carlo run samples uniformly among requested<br>&#10;  reference years admissible for the studied year<br>&#10;  (<code>reference_year &lt;= year</code>). The same sampled reference year is<br>&#10;  shared across the run when admissible; years for which it is not<br>&#10;  admissible resample among their admissible reference years.<br>&#10;&#160;<br>&#10;  Nested key:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether reference year uncertainty is active.</span><br>&#10;&#160;<br>&#10;<span style="color:#c45f00">&bull; <code>inter_mrio_uncertainty</code>: optional source block. To activate<br>&#10;  it, write <code>{&quot;active&quot;: True, &quot;alternate_source&quot;: &quot;&lt;disaggregated label&gt;&quot;}</code>, for example<br>&#10;  <code>{&quot;active&quot;: True, &quot;alternate_source&quot;: &quot;oecd_electricity&quot;}</code>.<br>&#10;  It applies continuous uniform interpolation between the main<br>&#10;  MRIO source and an alternate published disaggregated aSoCC<br>&#10;  source created by <code>disaggregate_asocc(...)</code>. It applies only<br>&#10;  to non LCIA methods.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#c45f00">  &bull; <code>active</code>: Whether inter MRIO uncertainty is active.</span><br>&#10;<span style="color:#c45f00">  &bull; <code>alternate_source</code>: Published disaggregated aSoCC source</span><br>&#10;    label used as the alternate MRIO source.<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>inter_method_uncertainty</code>: optional source block. It<br>&#10;  <strong>defaults to</strong> <code>{&quot;active&quot;: True, &quot;mode&quot;: &quot;equal_weight&quot;}</code>.<br>&#10;  Each Monte Carlo run samples one method leaf among the selected<br>&#10;  deterministic and external methods. Equal weight mode writes the<br>&#10;  tree CSV, README, and rendered probability tree under the<br>&#10;  run folder <code>figures/inter_method_tree/</code>. To prepare custom<br>&#10;  weights before running uncertainty, use<br>&#10;  <code>write_asocc_weight_template(...)</code>; it writes<br>&#10;  <code>equal_weights.csv</code>, <code>README_inter_method_weights.txt</code>, and<br>&#10;  <code>probability_tree__equal_weights.&lt;ext&gt;</code> under<br>&#10;  <code>B1_asocc/preview_inter_method_weights/</code>. Use<br>&#10;  <code>preview_asocc_weight_tree(...)</code> to validate and render a<br>&#10;  custom probability tree before using<br>&#10;  <code>{&quot;mode&quot;: &quot;custom&quot;, &quot;version_name&quot;: &quot;...&quot;}</code>.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether inter method uncertainty is active.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>mode</code>: Inter method sampling mode. Accepted values are</span><br>&#10;<span style="color:#087f5b">    <code>&quot;equal_weight&quot;</code></span> and <code>&quot;custom&quot;</code>.<br>&#10;  &bull; <code>version_name</code>: Custom weight version used when<br>&#10;    <code>mode=&quot;custom&quot;</code>.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>sobol_parameters</code></td><td>Sobol sensitivity settings. Sobol analysis estimates<br>&#10;the contribution of active uncertainty sources to output variance<br>&#10;and writes <code>README_sobol.txt</code> under <code>results/sobol/</code> for<br>&#10;interpretation. The <strong>default</strong> has <code>active=True</code> and runs Sobol in<br>&#10;convergence mode. Sobol base sizes must be powers of two.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>active</code>: Whether Sobol sensitivity analysis is active.</span><br>&#10;<span style="color:#087f5b">&bull; <code>convergence</code>: convergence mode block. When Sobol is active,<br>&#10;  exactly one of <code>convergence</code> or <code>fixed</code> is active.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether convergence mode is active.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>max_base_samples</code>: Maximum Sobol base size.</span><br>&#10;<span style="color:#087f5b">  &bull; <code>rtol</code>: Relative tolerance for monitored <code>S1</code> and <code>ST</code><br>&#10;    indices.</span><br>&#10;&#160;<br>&#10;<span style="color:#c45f00">&bull; <code>fixed</code>: fixed Sobol base size mode block.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>active</code>: Whether fixed mode is active.</span><br>&#10;  &bull; <code>n_base_samples</code>: Exact Sobol base size.<br>&#10;&#160;<br>&#10;&bull; <code>sobol_years</code>: Studied output years evaluated by Sobol. When<br>&#10;  <strong>omitted</strong>, Sobol evaluates only the first and last studied years<br>&#10;  in the requested studied year set.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>external_method</code></td><td>Optional external aSoCC method selector. Use<br>&#10;<code>{&quot;l1_methods&quot;: [...]}</code> for L1 functional units. For L2<br>&#10;functional units use <code>{&quot;one_step_methods&quot;: [...]}</code> and/or<br>&#10;<code>{&quot;l1_l2_pairs&quot;: [&quot;&lt;l1_method&gt;::&lt;l2_method&gt;&quot;, ...]}</code>.<br>&#10;<strong>Omit</strong> this argument or pass <code>None</code> when using only native pyaesa<br>&#10;deterministic aSoCC methods. Use <code>prepare_external_inputs(...)</code><br>&#10;to import the external aSoCC runnable CSV examples and README guide, then<br>&#10;follow the imported guide for external method names, selector<br>&#10;syntax, and deterministic or Monte Carlo external aSoCC CSV<br>&#10;preparation.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Public uncertainty table format, either<br>&#10;<code>&quot;csv_compact&quot;</code> or <code>&quot;parquet&quot;</code>. <strong>Defaults to</strong><br>&#10;<code>&quot;csv_compact&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull; <code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_options</code></td><td>Figure product selector mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;per_method&quot;: True, &quot;multi_method&quot;: True, &quot;inter_method&quot;: True}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>per_method</code>: Whether to render method specific figures, with one separate<br>&#10;  figure for each allocation method.<br>&#10;&#160;<br>&#10;&bull; <code>multi_method</code>: Whether to render cross method comparison figures, with<br>&#10;  multiple allocation methods shown in the same figure.<br>&#10;&#160;<br>&#10;&bull; <code>inter_method</code>: Whether to render inter method uncertainty figures. These<br>&#10;  figures use the same method specific layout as <code>per_method</code>, but represent<br>&#10;  uncertainty induced by the inter method uncertainty setting rather than<br>&#10;  comparing individual allocation methods. This option is ignored when inter<br>&#10;  method uncertainty is inactive.</td></tr>
<tr><td colspan="2" style="height:0.75rem;"></td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, refresh both the resolved deterministic aSoCC prerequisite and the resolved aSoCC Monte Carlo outputs for this uncertainty request. The deterministic refresh removes the source and version <code>deterministic</code> output scope described for <code>deterministic_asocc(...)</code>. The Monte Carlo refresh removes matching run folders for the current request under the adjacent <code>monte_carlo</code> root. External aSoCC inputs, processed MRIO inputs, processed population and GDP, raw downloads, and downstream aCC or ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>



### aSoCC Monte Carlo with LCIA uncertainty added and Sobol indices, using defaults where omitted

In [ ]:
from pyaesa import uncertainty_asocc

uncertainty_asocc(
    base_asocc_args={
        "project_name": project_name_,
        "source": source_,
        "years": study_period_,
        "fu_code": fu_code_,
        "s_p": s_p_,
        "r_c": r_c_,
        "lcia_method": lcia_method_,
    },
    uncertainty_config={
        "mc_parameters": {
            "fixed": {"active": True, "n_runs": 20000},
            "convergence": {"active": False},
        },
        "lcia_uncertainty": {
            "active": True,
            "sector_cov_mapping": {"Paper": "Paper"},
        },
    },
    sobol_parameters={
        "active": True,
        "fixed": {"active": True, "n_base_samples": 128},
        "convergence": {"active": False},
    },
    refresh=True,
)

# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at `tutorials/study_objectives/...`

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available includes:
- inter-method uncertainty with custom weights
- inter-MRIO uncertainty
- projection uncertainty
- reference year uncertainty

### Inter-method uncertainty with a custom aSoCC

Inter MRIO uncertainty with a custom aSoCC can be implemented through the `external_method` parameter.

<p style="color:#b00020"><strong>IN PROGRESS:</strong> Example available shortly! </p>

### Inter-method uncertainty with custom weights between allocation methods

Inter method custom weights are prepared with
`write_asocc_weight_template(...)` and documented in
`README_inter_method_weights.txt`.

### Inter-MRIO uncertainty

Inter MRIO uncertainty is inactive by default and applies only to non LCIA
methods with an alternate source. It can be included through the `inter_mrio_uncertainty` parameter. 

<p style="color:#b00020"><strong>IN PROGRESS:</strong> Example available shortly! </p>

### Projection uncertainty and reference year uncertainty

`reference_year_uncertainty` and 
`projection_uncertainty`

<p style="color:#b00020"><strong>IN PROGRESS:</strong> Example available shortly! </p>